In [34]:
import pandas as pd

df = pd.read_csv(
    "global_pharmacy_sales_2020_2025_daily_dataset.csv"
)

df.head()

,date,year,month,day,region,country,category,medicine,age_group,units_sold,unit_price,stock_level,expiry_days_remaining,covid_flag
0,2025-10-12,2025,10,12,Middle East,UAE,Chronic,Amlodipine,0-12,328,45.38,4774,466,0
1,2020-09-15,2020,9,15,South Asia,India,Chronic,Amlodipine,26-45,371,75.49,4584,181,1
2,2020-02-26,2020,2,26,Middle East,UAE,Vitamin,Vitamin C,0-12,948,22.51,3934,556,1
3,2025-11-09,2025,11,9,South Asia,Sri Lanka,Chronic,Amlodipine,0-12,275,63.29,4544,330,0
4,2022-04-04,2022,4,4,South America,Argentina,Chronic,Amlodipine,26-45,563,44.28,4284,590,0


In [36]:
medicine_df = df[
    ['medicine', 'category', 'age_group']
].drop_duplicates()

print(medicine_df.head())

       medicine    category age_group
0    Amlodipine     Chronic      0-12
1    Amlodipine     Chronic     26-45
2     Vitamin C     Vitamin      0-12
5     Metformin     Chronic     26-45
6  Azithromycin  Antibiotic       65+


In [37]:
medicine_df['text'] = (
    medicine_df['category'] + " " +
    medicine_df['age_group']
)

In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

vectors = vectorizer.fit_transform(
    medicine_df['text']
)

In [48]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [53]:
def recommend(medicine_name, top_k=5):

    # Check if medicine exists
    if medicine_name not in medicine_df['medicine'].values:
        print("Medicine not found")
        return

    # Get index of medicine
    idx = medicine_df[
        medicine_df['medicine'] == medicine_name
    ].index[0]

    # Get similarity scores
    scores = list(enumerate(similarity[idx]))

    # Sort by similarity score
    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Store recommendations
    recommendations = []

    for i, score in scores[1:top_k+1]:

        recommendations.append({
            "medicine": medicine_df.iloc[i]['medicine'],
            "category": medicine_df.iloc[i]['category'],
            "age_group": medicine_df.iloc[i]['age_group'],
            "similarity_score": round(score, 3)
        })

    return recommendations

In [56]:
results = recommend("Paracetamol")

for item in results:
    print(item)

{'medicine': 'Cough Syrup', 'category': 'Cough_Cold', 'age_group': '26-45', 'similarity_score': np.float64(1.0)}
{'medicine': 'Amlodipine', 'category': 'Chronic', 'age_group': '26-45', 'similarity_score': np.float64(0.667)}
{'medicine': 'Metformin', 'category': 'Chronic', 'age_group': '26-45', 'similarity_score': np.float64(0.667)}
{'medicine': 'Vitamin C', 'category': 'Vitamin', 'age_group': '26-45', 'similarity_score': np.float64(0.667)}
{'medicine': 'Azithromycin', 'category': 'Antibiotic', 'age_group': '26-45', 'similarity_score': np.float64(0.667)}


In [57]:
recommend("ABC Medicine")

Medicine not found


In [58]:
import pickle

pickle.dump(
    vectorizer,
    open("vectorizer.pkl", "wb")
)

In [59]:
import pickle

pickle.dump(
    similarity,
    open("similarity_matrix.pkl", "wb")
)

print("Similarity matrix saved successfully!")

Similarity matrix saved successfully!


In [62]:
medicine_df.to_csv(
    "medicine_metadata.csv",
    index=False
)

In [64]:
from google.colab import files

files.download("vectorizer.pkl")
files.download("similarity_matrix.pkl")
files.download("medicine_metadata.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>